# Güç Ölçümü A: Boşta

In [5]:
from pynq import Overlay
ol = Overlay('empty.bit')

 # Güç Ölçümü B: Donanım kuruldu ama akış yok

In [16]:
#CFG    = "W_int8_A8"
CFG    = "W_int4_A8"
#CFG    = "A_int8_A4"
#CFG    = "A_int4_A4"
#CFG    = "W_ternary_A8"
BIT    = f"{CFG}.bit"          # .hwh aynı isimle yanında olmalı
GOLDEN = f"golden_{CFG}.npz"

In [17]:
from pynq import Overlay
ol = Overlay(BIT)   # zaten yüklü olanı yeniden yükler

# Güç Ölçümü C: Donanım Üzerinde akış

In [18]:
import numpy as np, time
from pynq import Overlay, allocate




ol  = Overlay(BIT)
ip  = ol.cnn_hw_0
print(ip.register_map)                       # ilk çalıştırmada bak, alan adlarını doğrula

g   = np.load(GOLDEN)
X, GC, GL = g["x"], g["cls"], g["logit"]
N, L = X.shape
NC   = GL.shape[1]

xb = allocate(shape=(N*L,),      dtype=np.int8)
yb = allocate(shape=(N*(NC+1),), dtype=np.int32)
xb[:] = X.reshape(-1); xb.flush()

def run(n_win):
    ip.register_map.x_in_1  = xb.device_address
    ip.register_map.y_out_1 = yb.device_address
    ip.register_map.n_win   = n_win
    ip.register_map.CTRL.AP_START = 1
    while ip.register_map.CTRL.AP_DONE == 0: pass
    yb.invalidate()

# --- 1. bit-exact doğrulama ---
run(N)
out = yb[:N*(NC+1)].reshape(N, NC+1)
mc  = int((out[:,0]  != GC).sum())
ml  = int((out[:,1:] != GL.astype(np.int32)).sum())
print(f"\nHW=golden  argmax uyumsuz {mc}/{N}   logit uyumsuz {ml}/{N*NC}")
print(f"HW doğruluk {(out[:,0]==g['y']).mean():.4f}")

# --- 2. tek pencere gecikmesi ---
t = []
for _ in range(200):
    s = time.perf_counter(); run(1); t.append(time.perf_counter()-s)
t = np.array(t)*1e3
print(f"tek pencere: {t.mean():.3f} ± {t.std():.3f} ms  (medyan {np.median(t):.3f})")

# --- 3. sürekli akış throughput ---
s = time.perf_counter(); run(N); d = time.perf_counter()-s
print(f"toplu {N} pencere: {1e3*d:.1f} ms -> {N/d:.1f} pencere/s, "
      f"{1e3*d/N:.3f} ms/pencere")

# --- 4. güç ölçümü için kesintisiz döngü ---
SEC = 5
print(f"\n>>> {SEC} s kesintisiz çıkarım başlıyor, kaydı ŞİMDİ başlat")
n, s = 0, time.perf_counter()
while time.perf_counter() - s < SEC:
    run(N); n += N
print(f">>> bitti: {n} pencere, {n/(time.perf_counter()-s):.1f} pencere/s. Kaydı durdur.")

RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  x_in_1 = Register(x_in=0),
  x_in_2 = Register(x_in=0),
  y_out_1 = Register(y_out=0),
  y_out_2 = Register(y_out=0),
  n_win = Register(n_win=0)
}

HW=golden  argmax uyumsuz 0/1534   logit uyumsuz 0/15340
HW doğruluk 0.9433
tek pencere: 2.307 ± 0.144 ms  (medyan 2.302)
toplu 1534 pencere: 2817.8 ms -> 544.4 pencere/s, 1.837 ms/pencere

>>> 5 s kesintisiz çıkarım başlıyor, kaydı ŞİMDİ başlat
>>> bitti: 3068 pencere, 544.4 pencere/s. Kaydı durdur.


In [3]:
import numpy as np
def macro_f1(y_true, y_pred, n_cls=10):
    f = []
    for c in range(n_cls):
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())
        f.append(0.0 if tp == 0 else 2*tp / (2*tp + fp + fn))
    return float(np.mean(f))

print(f"HW macro-F1 {macro_f1(g['y'], out[:,0]):.4f}   "
      f"(accuracy {(out[:,0]==g['y']).mean():.4f})")

HW macro-F1 0.8806   (accuracy 0.9270)
